# Crop AP-Ratio Prediction
### Feature Engineering → CSV Export → Regression Modelling

**Splits**
| Strategy | Description |
|---|---|
| Random 70/15/15 | Standard i.i.d. baseline |
| Spatial OOD | Disjoint district partitions - tests geographic generalisation |
| Season OOD | Season-as-time (Rabi → Kharif 1 → Kharif 2) - tests seasonal generalisation |

**Dataset:** Bangladesh crop production (4190 rows · 15 columns · 64 districts · 3 seasons · 72 crops)


## Notebook Role

This is the main model-comparison notebook. CatBoost has been integrated here so `catboost_apply_init.ipynb` can remain only as a reference/outlier-investigation draft. The core results should be reported from this notebook after rerunning it top-to-bottom.

Important terminology: because SPAS-Dataset-BD does not contain a year column, the former "temporal" split is treated as a **Season OOD** split, not true year-based temporal forecasting.


## 1. Imports

In [ ]:
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from copy import deepcopy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, median_absolute_error)
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", None)
SEED = 42
np.random.seed(SEED)
print("All imports OK")


## 2. Load Data

In [ ]:
DATA_PATH = "../dataset/preprocessed_data.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Shape : {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(3)


In [ ]:
print("dtypes")
print(df_raw.dtypes)
print()
print("nulls")
print(df_raw.isnull().sum())
print()
print("Season values", df_raw["Season"].unique())
print("Districts", df_raw["District"].nunique(), "unique")
print("Crops", df_raw["Crop Name"].nunique(), "unique")
print()
df_raw.describe()


## 3. Feature Engineering

| Group | Features |
|---|---|
| **Thermal** | Diurnal range, temp mean, GDD proxy (base 10°C), Magnus SVP, VPD |
| **Humidity** | Mid-range humidity, humidity range (already present) |
| **Crop calendar** | Transplant / growth-start / harvest as month ordinals |
| **Season temporal** | Rabi=0 → Kharif 1=1 → Kharif 2=2 (agronomic time axis) |
| **Season cyclic** | sin/cos encoding of season ordinal |
| **Interactions** | Area × diurnal range, Avg Temp × Avg Humidity, log(Area) |
| **Categorical** | Label-encoded Season, Crop Name, Transplant, Growth, Harvest, District |

> **No year column exists** in the dataset. The temporal axis is the agronomic  
> season order: Rabi (winter/dry) → Kharif 1 (pre-monsoon) → Kharif 2 (monsoon).


In [ ]:
df = df_raw.copy()

# ── Month mapping ─────────────────────────────────────────────────────────────
MONTH_MAP = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5,  "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
}

def first_month_ordinal(s: str) -> float:
    """Return the first month number found in a string like 'Jul to Oct'."""
    if pd.isna(s):
        return np.nan
    for tok in str(s).split():
        tok = tok.strip().rstrip(",")
        if tok in MONTH_MAP:
            return float(MONTH_MAP[tok])
    return np.nan

df["transplant_month"] = df["Transplant"].apply(first_month_ordinal)
df["growth_start_month"] = df["Growth"].apply(first_month_ordinal)
df["harvest_start_month"] = df["Harvest"].apply(first_month_ordinal)

# Fill NaN month values with median of each month column
df["transplant_month"] = df["transplant_month"].fillna(df["transplant_month"].median())
df["growth_start_month"] = df["growth_start_month"].fillna(df["growth_start_month"].median())
df["harvest_start_month"] = df["harvest_start_month"].fillna(df["harvest_start_month"].median())

print("Month columns after filling NaN:")
print(f"  transplant_month NaN: {df['transplant_month'].isnull().sum()}")
print(f"  growth_start_month NaN: {df['growth_start_month'].isnull().sum()}")
print(f"  harvest_start_month NaN: {df['harvest_start_month'].isnull().sum()}")

# ── Thermal features ──────────────────────────────────────────────────────────
df["temp_diurnal_range"] = df["Max Temp"] - df["Min Temp"]
df["temp_mean"] = (df["Max Temp"] + df["Min Temp"]) / 2
df["gdd_proxy"] = (df["temp_mean"] - 10).clip(lower=0)   # base 10°C

# Magnus approximation: saturation vapour pressure (kPa)
df["svp_proxy"] = 0.611 * np.exp(17.27 * df["Avg Temp"] / (df["Avg Temp"] + 237.3))
# Vapour pressure deficit
df["vpd_proxy"] = df["svp_proxy"] * (1 - df["Avg Humidity"] / 100)

# ── Humidity features ─────────────────────────────────────────────────────────
df["humidity_mid"] = (df["Humidity Max"] + df["Humidity Min"]) / 2

# ── Season order axis (agronomic order, no year available) ─────────────────
# Rabi = winter/dry season → earliest  (maps to 0)
# Kharif 1 = pre-monsoon → mid  (maps to 1)
# Kharif 2 = monsoon → latest  (maps to 2)
SEASON_ORDER = {"Rabi": 0, "Kharif 1": 1, "Kharif 2": 2}
df["season_order"] = df["Season"].map(SEASON_ORDER)

# Cyclic encoding so model sees season as circular
df["season_sin"] = np.sin(2 * np.pi * df["season_order"] / 3)
df["season_cos"] = np.cos(2 * np.pi * df["season_order"] / 3)

# ── Interaction features ──────────────────────────────────────────────────────
df["area_log"] = np.log1p(df["Area"])
df["area_x_diurnal"] = df["Area"] * df["temp_diurnal_range"]
df["temp_x_humidity"] = df["Avg Temp"] * df["Avg Humidity"]

# ── Label encode categoricals ─────────────────────────────────────────────────
CAT_COLS = ["Season", "Crop Name", "Transplant", "Growth", "Harvest"]
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

le_district = LabelEncoder()
df["District_enc"] = le_district.fit_transform(df["District"].astype(str))

print(f"Feature engineering complete. New shape: {df.shape}")
df.head()


## 4. Export Engineered Dataset

In [ ]:
OUT_CSV = "../dataset/crop_engineered.csv"

# Keep District + Season for split logic alongside all features and target
meta_cols = ["District", "Season", "season_order"]
engineered_df = pd.concat([df[meta_cols],
                           df.drop(columns=meta_cols + list(df_raw.columns) + ["District_enc"])
                             .assign(**{"District_enc": df["District_enc"]}),
                           df["AP Ratio"]], axis=1)

# Cleaner: just dump everything useful
export_cols = (meta_cols +
               [c for c in df.columns if c not in df_raw.columns and c not in meta_cols])
engineered_df = df[export_cols + ["AP Ratio"]].copy()
engineered_df.to_csv(OUT_CSV, index=False)

print(f"Saved → {OUT_CSV}")
print(f"Shape : {engineered_df.shape[0]} rows × {engineered_df.shape[1]} cols")
engineered_df.head(3)


## 5. Feature Matrix & Target

In [ ]:
TARGET = "AP Ratio"

FEATURE_COLS = [
    # Raw numeric
    "Area", "area_log", "Avg Temp", "Avg Humidity",
    "Max Temp", "Min Temp", "Humidity Min", "Humidity Max", "Humidity Range",
    # Thermal
    "temp_diurnal_range", "temp_mean", "gdd_proxy", "svp_proxy", "vpd_proxy",
    # Humidity
    "humidity_mid",
    # Crop calendar
    "transplant_month", "growth_start_month", "harvest_start_month",
    # Season
    "season_order", "season_sin", "season_cos",
    # Interactions
    "area_x_diurnal", "temp_x_humidity",
    # Encoded categoricals
    "Season_enc", "Crop Name_enc",
    "Transplant_enc", "Growth_enc", "Harvest_enc",
    "District_enc",
]

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

print(f"Feature matrix : {X.shape}")
print(f"Target - min: {y.min():.4f}  max: {y.max():.4f}  mean: {y.mean():.4f}  std: {y.std():.4f}")


In [ ]:
# Target distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(y, bins=60, color="steelblue", edgecolor="white")
ax1.set_title("AP Ratio Distribution")
ax1.set_xlabel("AP Ratio")
ax1.set_ylabel("Count")

ax2.hist(np.log1p(y), bins=60, color="coral", edgecolor="white")
ax2.set_title("log1p(AP Ratio) Distribution")
ax2.set_xlabel("log1p(AP Ratio)")

plt.suptitle("Target Variable", fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Train / Validation / Test Splits

| Split | Train | Val | Test | Design |
|---|---|---|---|---|
| **Random** | 70% | 15% | 15% | i.i.d. rows |
| **Spatial OOD** | 70% districts | 15% districts | 15% districts | Disjoint district sets |
| **Season OOD** | Rabi (~28%) | Kharif 1 (~34%) | Kharif 2 (~38%) | Agronomic season order |

The Season OOD proportions deviate from 70/15/15 by design — each season is a  
fully disjoint temporal partition mirroring climate-aware forecasting evaluations.


In [ ]:
SEED = 42

# A. Random 70 / 15 / 15
X_tr_r, X_tmp, y_tr_r, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val_r, X_te_r, y_val_r, y_te_r = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED)

print("Random Split")
print(f"  Train : {len(X_tr_r):>4}  ({len(X_tr_r)/len(X)*100:.1f}%)")
print(f"  Val   : {len(X_val_r):>4}  ({len(X_val_r)/len(X)*100:.1f}%)")
print(f"  Test  : {len(X_te_r):>4}  ({len(X_te_r)/len(X)*100:.1f}%)")


In [ ]:
# B. Spatial OOD — disjoint districts
all_districts = df["District"].unique().copy()
np.random.shuffle(all_districts)

n_d = len(all_districts)
n_tr_d = int(0.70 * n_d)
n_val_d = int(0.15 * n_d)

train_districts = set(all_districts[:n_tr_d])
val_districts = set(all_districts[n_tr_d : n_tr_d + n_val_d])
test_districts = set(all_districts[n_tr_d + n_val_d :])

mask_tr_s = df["District"].isin(train_districts)
mask_val_s = df["District"].isin(val_districts)
mask_te_s = df["District"].isin(test_districts)

X_tr_s, y_tr_s = X[mask_tr_s.values], y[mask_tr_s.values]
X_val_s, y_val_s = X[mask_val_s.values], y[mask_val_s.values]
X_te_s, y_te_s = X[mask_te_s.values], y[mask_te_s.values]

print("Spatial OOD Split")
print(f"  Train districts : {len(train_districts)}  →  {len(X_tr_s):>4} rows  ({len(X_tr_s)/len(X)*100:.1f}%)")
print(f"  Val   districts : {len(val_districts)}   →  {len(X_val_s):>4} rows  ({len(X_val_s)/len(X)*100:.1f}%)")
print(f"  Test  districts : {len(test_districts)}   →  {len(X_te_s):>4} rows  ({len(X_te_s)/len(X)*100:.1f}%)")


In [ ]:
# C. Season OOD — season-shift
#
# Bangladesh agronomic calendar (no year column in dataset):
#   Rabi     (winter/dry,   Nov–Mar)  → EARLIEST  → Train
#   Kharif 1 (pre-monsoon, Mar–Jul)   → MID       → Validation
#   Kharif 2 (monsoon,     Jun–Nov)   → LATEST    → Test
#
# The model is trained on dry-season crops and evaluated on monsoon-season crops -
# a genuine distributional shift in temperature, humidity, and crop profile.

mask_tr_season  = df["season_order"] == 0   # Rabi
mask_val_season = df["season_order"] == 1   # Kharif 1
mask_te_season  = df["season_order"] == 2   # Kharif 2

X_tr_season, y_tr_season  = X[mask_tr_season.values], y[mask_tr_season.values]
X_val_season, y_val_season = X[mask_val_season.values], y[mask_val_season.values]
X_te_season, y_te_season  = X[mask_te_season.values], y[mask_te_season.values]

print("Season OOD Split (Season-Shift)")
print(f"  Train - Rabi (season_order=0) : {len(X_tr_season):>4} rows  ({len(X_tr_season)/len(X)*100:.1f}%)")
print(f"  Val - Kharif 1 (season_order=1) : {len(X_val_season):>4} rows  ({len(X_val_season)/len(X)*100:.1f}%)")
print(f"  Test - Kharif 2 (season_order=2) : {len(X_te_season):>4} rows  ({len(X_te_season)/len(X)*100:.1f}%)")


In [ ]:
# Split size visualisation
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
split_info = [
    ("Random", [len(X_tr_r), len(X_val_r), len(X_te_r)]),
    ("Spatial OOD", [len(X_tr_s), len(X_val_s), len(X_te_s)]),
    ("Season OOD", [len(X_tr_season), len(X_val_season), len(X_te_season)]),
]
colors = ["#4C72B0", "#DD8452", "#55A868"]
labels = ["Train", "Val", "Test"]

for ax, (title, sizes) in zip(axes, split_info):
    bars = ax.bar(labels, sizes, color=colors)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Rows")
    for bar, sz in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(sz), ha="center", va="bottom", fontsize=10)

plt.suptitle("Split Size Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Feature Scaling

In [ ]:
def scale_split(X_tr, X_val, X_te):
    """Fit scaler on train only; transform val and test."""
    scaler = StandardScaler()
    Xs_tr  = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    Xs_val = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
    Xs_te  = pd.DataFrame(scaler.transform(X_te), columns=X_te.columns, index=X_te.index)
    return Xs_tr, Xs_val, Xs_te, scaler

Xs_tr_r, Xs_val_r, Xs_te_r, scaler_r = scale_split(X_tr_r, X_val_r, X_te_r)
Xs_tr_s, Xs_val_s, Xs_te_s, scaler_s = scale_split(X_tr_s, X_val_s, X_te_s)
Xs_tr_season, Xs_val_season, Xs_te_season, scaler_t = scale_split(X_tr_season, X_val_season, X_te_season)

print("Scaling complete - fitted on each split's training fold independently.")


## 8. Model Training & Evaluation

In [ ]:
def compute_metrics(y_true, y_pred, label=""):
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"    {label:<8}  MAE={mae:.4f}  MedAE={medae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return dict(MAE=mae, MedAE=medae, RMSE=rmse, R2=r2)

def train_and_eval(name, model, Xs_tr, y_tr, Xs_val, y_val, Xs_te, y_te):
    model.fit(Xs_tr, y_tr)
    print(f"  {name}")
    val_m = compute_metrics(y_val, model.predict(Xs_val), "Val")
    test_m = compute_metrics(y_te, model.predict(Xs_te), "Test")
    return model, val_m, test_m

MODELS = {
    "Ridge" : Ridge(alpha=1.0),
    "RF" : RandomForestRegressor(n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1),
    "ExtraTrees" : ExtraTreesRegressor(n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1),
    "GBM" : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=SEED),
    "XGBoost" : XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=0),
    "LightGBM" : LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63, random_state=SEED, verbosity=-1),
    "CatBoost" : CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0, loss_function="RMSE", random_seed=SEED, allow_writing_files=False, verbose=False),
}


### 8.1 Random Split

In [ ]:
# Check for NaN values after feature engineering
nan_summary = X.isnull().sum()
print(nan_summary)
nan_cols = nan_summary[nan_summary > 0]

if len(nan_cols) > 0:
    print("Columns with NaN values:")
    print(nan_summary[nan_summary > 0])
    print(f"\nTotal NaN cells: {X.isnull().sum().sum()}")
else:
    print("No NaN values found!")

print(X.head())

In [ ]:
print("RANDOM SPLIT")
random_results = {}
for name, model in MODELS.items():
    m, val_m, test_m = train_and_eval(name, deepcopy(model),
                                      Xs_tr_r, y_tr_r,
                                      Xs_val_r, y_val_r,
                                      Xs_te_r,  y_te_r)
    random_results[name] = dict(model=m, val=val_m, test=test_m)


### 8.2 Spatial OOD Split

In [ ]:
print("SPATIAL OOD SPLIT")
spatial_results = {}
for name, model in MODELS.items():
    m, val_m, test_m = train_and_eval(name, deepcopy(model),
                                      Xs_tr_s, y_tr_s,
                                      Xs_val_s, y_val_s,
                                      Xs_te_s,  y_te_s)
    spatial_results[name] = dict(model=m, val=val_m, test=test_m)


### 8.3 Season OOD Split

In [ ]:
print("SEASON OOD SPLIT (Rabi → Kharif 1 → Kharif 2)")
season_results = {}
for name, model in MODELS.items():
    m, val_m, test_m = train_and_eval(name, deepcopy(model),
                                      Xs_tr_season, y_tr_season,
                                      Xs_val_season, y_val_season,
                                      Xs_te_season,  y_te_season)
    season_results[name] = dict(model=m, val=val_m, test=test_m)


## 9. Consolidated Results

In [ ]:
def to_df(results_dict, split_name):
    rows = []
    for name, d in results_dict.items():
        for stage in ("val", "test"):
            rows.append({"Split": split_name, "Model": name,
                         "Stage": stage.capitalize(), **d[stage]})
    return pd.DataFrame(rows)

all_results = pd.concat([
    to_df(random_results, "Random"),
    to_df(spatial_results, "Spatial OOD"),
    to_df(season_results, "Season OOD"),
], ignore_index=True)

for c in ["MAE", "MedAE", "RMSE", "R2"]:
    all_results[c] = all_results[c].round(4)

all_results = all_results.sort_values(["Split", "Stage", "RMSE"]).reset_index(drop=True)
print(all_results.to_string(index=False))


In [ ]:
# Best model per split on Test set
test_df = all_results[all_results["Stage"] == "Test"]
print("Best model per split (Test R²)")
for split in ["Random", "Spatial OOD", "Season OOD"]:
    best = test_df[test_df["Split"] == split].sort_values("R2", ascending=False).iloc[0]
    print(f"  {split:<15}  {best['Model']:<12}  R²={best['R2']:.4f}  RMSE={best['RMSE']:.4f}  MAE={best['MAE']:.4f}")


## 10. Visualisations

### 10.1 R² and RMSE Across All Splits

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
splits = ["Random", "Spatial OOD", "Season OOD"]
palette = sns.color_palette("Set2", len(MODELS))

for col, split in enumerate(splits):
    sub = test_df[test_df["Split"] == split].sort_values("R2", ascending=False)

    # R² row
    ax = axes[0, col]
    bars = ax.bar(sub["Model"], sub["R2"], color=palette)
    ax.set_title(f"{split}", fontweight="bold")
    ax.set_ylabel("Test R²" if col == 0 else "")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=35)
    for bar, val in zip(bars, sub["R2"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

    # RMSE row
    ax = axes[1, col]
    bars2 = ax.bar(sub["Model"], sub["RMSE"], color=palette)
    ax.set_ylabel("Test RMSE" if col == 0 else "")
    ax.tick_params(axis="x", rotation=35)
    for bar, val in zip(bars2, sub["RMSE"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Test R² (top) and RMSE (bottom) Across Split Strategies", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### 10.2 RMSE Heatmap (Model × Split)

In [ ]:
pivot_rmse = test_df.pivot_table(index="Model", columns="Split", values="RMSE")
pivot_r2 = test_df.pivot_table(index="Model", columns="Split", values="R2")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(pivot_rmse, annot=True, fmt=".4f", cmap="YlOrRd", ax=ax1)
ax1.set_title("Test RMSE - lower is better", fontweight="bold")

sns.heatmap(pivot_r2, annot=True, fmt=".4f", cmap="YlGn", ax=ax2)
ax2.set_title("Test R² - higher is better", fontweight="bold")

plt.suptitle("Model Performance Heatmaps", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()


### 10.3 Predicted vs Actual (Best Model, Each Split)

In [ ]:
def best_model_name(results, split_results):
    return (test_df[test_df["Split"] == results]
            .sort_values("R2", ascending=False)
            .iloc[0]["Model"])

splits_data = [
    ("Random", random_results, Xs_te_r, y_te_r),
    ("Spatial OOD", spatial_results, Xs_te_s, y_te_s),
    ("Season OOD", season_results, Xs_te_season, y_te_season),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (split_name, res_dict, Xs_te, y_te) in zip(axes, splits_data):
    bname = best_model_name(split_name, res_dict)
    y_pred = res_dict[bname]["model"].predict(Xs_te)
    r2 = r2_score(y_te, y_pred)

    ax.scatter(y_te, y_pred, alpha=0.35, s=15, color="steelblue")
    lims = [min(y_te.min(), y_pred.min()) - 0.05,
            max(y_te.max(), y_pred.max()) + 0.05]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect")
    ax.set_title(f"{split_name}\n{bname}  (R²={r2:.3f})", fontweight="bold")
    ax.set_xlabel("Actual AP Ratio")
    ax.set_ylabel("Predicted AP Ratio" if ax == axes[0] else "")
    ax.legend(fontsize=8)

plt.suptitle("Predicted vs Actual - Best Model per Split", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/pred_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()


### 10.4 Feature Importance - Best Model (Random Split)

In [ ]:
best_name_r = best_model_name("Random", random_results)
best_model_r = random_results[best_name_r]["model"]

if hasattr(best_model_r, "feature_importances_"):
    imp = pd.Series(best_model_r.feature_importances_, index=FEATURE_COLS)
    top20 = imp.nlargest(20).sort_values()

    fig, ax = plt.subplots(figsize=(9, 6))
    colors_imp = ["#2196F3" if "enc" not in f else "#FF9800" for f in top20.index]
    top20.plot(kind="barh", ax=ax, color=colors_imp)
    ax.set_title(f"Top-20 Feature Importances - {best_name_r} (Random Split)", fontweight="bold")
    ax.set_xlabel("Importance")

    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#2196F3", label="Engineered / Raw"), Patch(color="#FF9800", label="Categorical Encoded")], loc="lower right")
    plt.tight_layout()
    plt.savefig("../plots/feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"{best_name_r} does not expose feature_importances_")


### 10.5 Residual Analysis (Best Model, All Splits)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, (split_name, res_dict, Xs_te, y_te) in enumerate(splits_data):
    bname = best_model_name(split_name, res_dict)
    y_pred = res_dict[bname]["model"].predict(Xs_te)
    resid = y_te.values - y_pred

    # Residuals vs predicted
    ax = axes[0, col]
    ax.scatter(y_pred, resid, alpha=0.35, s=12, color="coral")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{split_name} - {bname}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual" if col == 0 else "")

    # Residual histogram
    ax = axes[1, col]
    ax.hist(resid, bins=40, color="slateblue", edgecolor="white")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count" if col == 0 else "")
    ax.set_title(f"μ={resid.mean():.4f}  σ={resid.std():.4f}")

plt.suptitle("Residual Analysis - Best Model per Split", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/residuals.png", dpi=150, bbox_inches="tight")
plt.show()


### 10.6 Generalisation Gap (Val R² vs Test R²)

In [ ]:
gap_rows = []
for split_name, res_dict in [("Random", random_results), ("Spatial OOD", spatial_results), ("Season OOD", season_results)]:
    for name, d in res_dict.items():
        gap_rows.append({
            "Split" : split_name,
            "Model" : name,
            "Val R²" : round(d["val"]["R2"], 4),
            "Test R²" : round(d["test"]["R2"], 4),
            "Gap (V-T)" : round(d["val"]["R2"] - d["test"]["R2"], 4),
        })

gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False))


In [ ]:
# Gap heatmap
pivot_gap = gap_df.pivot_table(index="Model", columns="Split", values="Gap (V-T)")

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_gap, annot=True, fmt=".4f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Generalisation Gap  (Val R² − Test R²)\nPositive = overfitting signal", fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/gen_gap.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. Summary & Recommendations

In [ ]:
print("FINAL MODEL RECOMMENDATIONS: ")

for split in ["Random", "Spatial OOD", "Season OOD"]:
    sub  = test_df[test_df["Split"] == split].sort_values("R2", ascending=False)
    best = sub.iloc[0]
    print(f"\n  [{split}]")
    print(f"    Best model : {best['Model']}")
    print(f"    Test R² : {best['R2']:.4f}")
    print(f"    Test RMSE : {best['RMSE']:.4f}")
    print(f"    Test MAE : {best['MAE']:.4f}")

print()
print(" Interpretation ")
print("  1. Random split reflects best-case i.i.d. performance.")
print("  2. Spatial OOD reflects ability to generalise to new geographic regions / unseen districts.")
print("  3. Season OOD reflects ability to generalise across seasons — model trained on Rabi (winter) crops is evaluated on Kharif 2 (monsoon) crops, a genuine distributional shift in climate and crop profile.")
print("  4. Large Random→OOD performance gap signals the model memorises district/season-specific patterns; consider adding more cross-district aggregation features.")
